## Path configuration

In [ ]:
from pathlib import Path
import os

PROJECT_NAME = "MALDIAlign"

cwd = Path().resolve()

# Walk upwards until we find the project folder
target = None
for parent in [cwd] + list(cwd.parents):
    if parent.name == PROJECT_NAME:
        target = parent
        break

# If the project folder is found and we are not already there, then change cwd
if target is not None and target != cwd:
    os.chdir(target)

print("Working directory:", os.getcwd())

## Imports

In [ ]:
import pickle
import numpy as np
import pandas as pd

# utils
from utils.config import load_config
from utils.data import load_pkl, map_domains
from utils.metrics import *
from utils.viz import *

# tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

## Data loading

In [ ]:
cfg = load_config()
driams_pkl = cfg["data"]["DRIAMS_REDUCED_PKL"]
marisma_pkl = cfg["data"]["MARISMa_REDUCED_PKL"]

In [ ]:
driams = load_pkl(driams_pkl)
marisma = load_pkl(marisma_pkl)

In [ ]:
data, label, meta = driams["data"], driams["label"], pd.DataFrame.from_records(list(driams["meta"]))

In [ ]:
# Apply normalization: scale each spectrum to [0, 1]
X_min = data.min(axis=1, keepdims=True)
X_max = data.max(axis=1, keepdims=True)
data_norm = (data - X_min) / (X_max - X_min + 1e-8)

In [ ]:
data_marisma, label_marisma, meta_marisma = marisma["data"], marisma["label"], pd.DataFrame.from_records(list(marisma["meta"]))

In [ ]:
meta_marisma = meta_marisma.copy()

meta_marisma.insert(
    loc=0,
    column="hospital",
    value="MARISMA"
)

In [ ]:
meta_marisma

In [ ]:
# Apply normalization: scale each spectrum to [0, 1]
X_min_marisma = data_marisma.min(axis=1, keepdims=True)
X_max_marisma = data_marisma.max(axis=1, keepdims=True)
data_norm_marisma = (data_marisma - X_min_marisma) / (X_max_marisma - X_min_marisma + 1e-8)

### All years

In [ ]:
meta

In [ ]:
# Filter the data by hospital
filtered_data = {}
for hosp in meta["hospital"].unique():
    idx = np.where(meta["hospital"].values == hosp)[0]
    filtered_data[hosp] = {
        "data": data_norm[idx],
        "label": label[idx],
        "meta": meta.iloc[idx]
    }

In [ ]:
for hosp in meta["hospital"].unique():
    print(f"Printing species in center {hosp}...")
    species, counts = np.unique(filtered_data[hosp]["label"], return_counts=True)
    for sp, n in zip(species, counts):
        print(f"{sp}: {n}")
    print("")

In [ ]:
# Declare the datasets
dataA, labelA, metaA = filtered_data["DRIAMS_A"]["data"], filtered_data["DRIAMS_A"]["label"], filtered_data["DRIAMS_A"]["meta"]
dataB, labelB, metaB = filtered_data["DRIAMS_B"]["data"], filtered_data["DRIAMS_B"]["label"], filtered_data["DRIAMS_B"]["meta"]
dataC, labelC, metaC = filtered_data["DRIAMS_C"]["data"], filtered_data["DRIAMS_C"]["label"], filtered_data["DRIAMS_C"]["meta"]
dataD, labelD, metaD = filtered_data["DRIAMS_D"]["data"], filtered_data["DRIAMS_D"]["label"], filtered_data["DRIAMS_D"]["meta"]

### 2018

In [ ]:
meta2018 = meta.iloc[np.where(meta["year"] == '2018')]

In [ ]:
mask_2018_m = meta_marisma["year"] == "2018"

meta2018_MARISMA = meta_marisma.loc[mask_2018_m].reset_index(drop=True)
data2018_MARISMA = data_norm_marisma[mask_2018_m.values]
label2018_MARISMA = label_marisma[mask_2018_m.values]

In [ ]:
meta2018

In [ ]:
meta2018_MARISMA

In [ ]:
# Filter the data by hospital in 2018
filtered_data2018 = {}
for hosp in meta2018["hospital"].unique():
    idx = np.where(meta2018["hospital"].values == hosp)[0]
    filtered_data2018[hosp] = {
        "data": data_norm[idx],
        "label": label[idx],
        "meta": meta2018.iloc[idx]
    }

In [ ]:
# Add MARISMA
filtered_data2018["MARISMA"] = {
    "data": data2018_MARISMA, 
    "label": label2018_MARISMA,
    "meta": meta2018_MARISMA
}

In [ ]:
filtered_data2018.keys()

In [ ]:
# Declare the datasets
dataA2018, labelA2018, metaA2018 = filtered_data2018["DRIAMS_A"]["data"], filtered_data2018["DRIAMS_A"]["label"], filtered_data2018["DRIAMS_A"]["meta"]
dataB2018, labelB2018, metaB2018 = filtered_data2018["DRIAMS_B"]["data"], filtered_data2018["DRIAMS_B"]["label"], filtered_data2018["DRIAMS_B"]["meta"]
dataC2018, labelC2018, metaC2018 = filtered_data2018["DRIAMS_C"]["data"], filtered_data2018["DRIAMS_C"]["label"], filtered_data2018["DRIAMS_C"]["meta"]
dataD2018, labelD2018, metaD2018 = filtered_data2018["DRIAMS_D"]["data"], filtered_data2018["DRIAMS_D"]["label"], filtered_data2018["DRIAMS_D"]["meta"]
data2018_MARISMA, label2018_MARISMA, meta2018_MARISMA = filtered_data2018["MARISMA"]["data"], filtered_data2018["MARISMA"]["label"], filtered_data2018["MARISMA"]["meta"]

In [ ]:
for hosp in filtered_data2018.keys():
    print(f"Printing species in center {hosp}...")
    species, counts = np.unique(filtered_data2018[hosp]["label"], return_counts=True)
    for sp, n in zip(species, counts):
        print(f"{sp}: {n}")
    print("")

### 2018 with common species

In [ ]:
# Obtener el conjunto de especies por hospital
species_per_hosp = {}

for hosp in filtered_data2018.keys():
    species_per_hosp[hosp] = set(filtered_data2018[hosp]["label"])

    print(f"{hosp}: {len(species_per_hosp[hosp])} species")

In [ ]:
# Intersección de especies comunes
common_species = set.intersection(*species_per_hosp.values())

print("\nCommon species across all hospitals (2018):")
print(f"Number of common species: {len(common_species)}")
print(sorted(common_species))

In [ ]:
# Filtrar cada hospital para quedarse solo con especies comunes
filtered_data2018_common = {}

for hosp in filtered_data2018.keys():
    labels_h = filtered_data2018[hosp]["label"]
    mask = np.isin(labels_h, list(common_species))

    filtered_data2018_common[hosp] = {
        "data": filtered_data2018[hosp]["data"][mask],
        "label": labels_h[mask],
        "meta": filtered_data2018[hosp]["meta"].iloc[mask]
    }

    print(
        f"{hosp}: {filtered_data2018_common[hosp]['data'].shape[0]} samples "
        f"after filtering"
    )

In [ ]:
# Re-declarar datasets finales (2018, especies comunes)
dataA2018_c, labelA2018_c, metaA2018_c = (
    filtered_data2018_common["DRIAMS_A"]["data"],
    filtered_data2018_common["DRIAMS_A"]["label"],
    filtered_data2018_common["DRIAMS_A"]["meta"],
)

dataB2018_c, labelB2018_c, metaB2018_c = (
    filtered_data2018_common["DRIAMS_B"]["data"],
    filtered_data2018_common["DRIAMS_B"]["label"],
    filtered_data2018_common["DRIAMS_B"]["meta"],
)

dataC2018_c, labelC2018_c, metaC2018_c = (
    filtered_data2018_common["DRIAMS_C"]["data"],
    filtered_data2018_common["DRIAMS_C"]["label"],
    filtered_data2018_common["DRIAMS_C"]["meta"],
)

dataD2018_c, labelD2018_c, metaD2018_c = (
    filtered_data2018_common["DRIAMS_D"]["data"],
    filtered_data2018_common["DRIAMS_D"]["label"],
    filtered_data2018_common["DRIAMS_D"]["meta"],
)

data_marisma_c, label_marisma_c, meta_marisma_c = (
    filtered_data2018_common["MARISMA"]["data"],
    filtered_data2018_common["MARISMA"]["label"],
    filtered_data2018_common["MARISMA"]["meta"],
)

In [ ]:
# Sanity checks finales
for hosp, (X, y) in {
    "A": (dataA2018_c, labelA2018_c),
    "B": (dataB2018_c, labelB2018_c),
    "C": (dataC2018_c, labelC2018_c),
    "D": (dataD2018_c, labelD2018_c),
    "MARISMA": (data_marisma_c, label_marisma_c)
}.items():
    assert set(np.unique(y)) == common_species
    print(f"Hospital {hosp}: OK — {X.shape[0]} samples")

## RF 2018, DRIAMS A against B-C-D-MARISMA, Common species

In [ ]:
X_train2018c, X_test2018c, y_train2018c, y_test2018c = train_test_split(
    dataA2018_c,
    labelA2018_c,
    test_size=0.2,
    stratify=labelA2018_c,
    shuffle=True,
    random_state=42
)

#### Train

In [ ]:
load_previous_rf = False
if not load_previous_rf:
    rf_final = RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42
    )
    #with open("models/baselines/trained_models/rf_baseline_normalized.pkl", "wb") as f:
    #    pickle.dump(rf_final, f)
    rf_final.fit(X_train2018c, y_train2018c)
else:
    print("Loading previous RF...")
    with open("models/baselines/trained_models/rf_baseline_normalized.pkl", "rb") as f:
        rf_final = pickle.load(f)

#### Test

In [ ]:
metricsA = metrics_report(X_test2018c, y_test2018c, rf_final, "DRIAMS-A")
metricsB = metrics_report(dataB2018_c, labelB2018_c, rf_final, "DRIAMS-B")
metricsC = metrics_report(dataC2018_c, labelC2018_c, rf_final, "DRIAMS-C")
metricsD = metrics_report(dataD2018_c, labelD2018_c, rf_final, "DRIAMS-D")
metricsE = metrics_report(data_marisma_c, label_marisma_c, rf_final, "MARISMA")

In [ ]:
print_metrics(metricsA)

In [ ]:
print_metrics(metricsB)

In [ ]:
print_metrics(metricsC)

In [ ]:
print_metrics(metricsD)

In [ ]:
print_metrics(metricsE)

Predict on test

In [ ]:
y_pred_A = rf_final.predict(X_test2018c)
y_pred_B = rf_final.predict(dataB2018_c)
y_pred_C = rf_final.predict(dataC2018_c)
y_pred_D = rf_final.predict(dataD2018_c)
y_pred_E = rf_final.predict(data_marisma_c)

idx_A = np.where(y_pred_A != y_test2018c)[0]
idx_B = np.where(y_pred_B != labelB2018_c)[0]
idx_C = np.where(y_pred_C != labelC2018_c)[0]
idx_D = np.where(y_pred_D != labelD2018_c)[0]
idx_E = np.where(y_pred_E != label_marisma_c)[0]

nA = len(labelA2018_c)
nB = len(labelB2018_c)
nC = len(labelC2018_c)
nD = len(labelD2018_c)
nE = len(label_marisma_c)

offset_A = 0
offset_B = nA
offset_C = nA + nB
offset_D = nA + nB + nC
offset_E = nA + nB + nC + nD

mis_idx_A_global = idx_A + offset_A
mis_idx_B_global = idx_B + offset_B
mis_idx_C_global = idx_C + offset_C
mis_idx_D_global = idx_D + offset_D
mis_idx_E_global = idx_E + offset_E

mis_idx_all_global = np.concatenate([
    mis_idx_A_global,
    mis_idx_B_global,
    mis_idx_C_global,
    mis_idx_D_global,
    mis_idx_E_global
])

#### t-SNEs

In [ ]:
load_global_pca = False
if not load_global_pca:
    pca_global = PCA(n_components=50, random_state=42)
    X_all_pca = pca_global.fit_transform(np.vstack([dataA2018_c, dataB2018_c, dataC2018_c, dataD2018_c, data_marisma_c]))
    #with open("models/baselines/trained_models/pca_global.pkl", "wb") as f:
    #    pickle.dump(pca_global, f)
else:
    with open("models/baselines/trained_models/pca_global.pkl", "rb") as f:
        pca_global = pickle.load(f)
        X_all_pca = pca_global.transform(data_norm)

In [ ]:
tsne_df = compute_tsne_df(X_all_pca, np.concatenate([labelA2018_c, labelB2018_c, labelC2018_c, labelD2018_c, label_marisma_c]), pd.concat([metaA2018_c, metaB2018_c, metaC2018_c, metaD2018_c, meta_marisma_c], ignore_index=True))

##### Global

In [ ]:
print("\n===== t-SNE: Global (colored by species) =====")
plot_tsne_global(tsne_df, per_species=False)

print("\n===== t-SNE: Global (colored by species) + misclassified points =====")
plot_tsne_global(tsne_df, per_species=False, idx=mis_idx_all_global)

print("\n===== t-SNE: Per-species (colored by hospital) =====")
plot_tsne_global(tsne_df, per_species=True)

print("\n===== t-SNE: Per-species (colored by hospital) + misclassified points =====")
plot_tsne_global(tsne_df, per_species=True, idx=mis_idx_all_global)

print("\n===== t-SNE: Per-species (overlay per hospital) + misclassified points =====")
plot_tsne_global(tsne_df, per_species=True, overlay_per_hospital=True, idx=mis_idx_all_global)

print("\n===== t-SNE: Per-species (overlay per year) =====")
plot_tsne_global(tsne_df, per_species=True, overlay_per_year=True)

print("\n===== t-SNE: Per-species (overlay per year) + misclassified points =====")
plot_tsne_global(tsne_df, per_species=True, overlay_per_year=True, idx=mis_idx_all_global)

##### Per species

In [ ]:
df_all, tsne_results = compute_tsne_per_species(X_all_pca, np.concatenate([labelA2018_c, labelB2018_c, labelC2018_c, labelD2018_c, label_marisma_c]), pd.concat([metaA2018_c, metaB2018_c, metaC2018_c, metaD2018_c, meta_marisma_c], ignore_index=True), prefix="f")

In [ ]:
print("\n===== t-SNE (per species): colored by hospital =====")
plot_tsne_species(df_all, tsne_results, idx=None)

print("\n===== t-SNE (per species): colored by hospital + misclassified points =====")
plot_tsne_species(df_all, tsne_results, idx=mis_idx_all_global)

print("\n===== t-SNE (per species): overlay per hospital + misclassified points =====")
plot_tsne_species(df_all, tsne_results, overlay_per_hospital=True, idx=mis_idx_all_global)

print("\n===== t-SNE (per species): overlay per year =====")
plot_tsne_species(df_all, tsne_results, overlay_per_year_per_species=True)

print("\n===== t-SNE (per species): overlay per year + misclassified points =====")
plot_tsne_species(df_all, tsne_results, overlay_per_year_per_species=True, idx=mis_idx_all_global)